# Phase 10 — Cross-Section Model & Backtest

Walk-forward expanding-window prediction on the full signal panel.  
Three models: **Ridge**, **ElasticNet**, **LightGBM**  
Long top-quintile / Short bottom-quintile, 10 bps/side TC, monthly rebalance.

In [ ]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from urbangrowth.config import data_path, get_pipeline

PROC_DIR    = data_path(get_pipeline()["processed_data_subdirs"].get("signal_tables", "processed/signals"))
HORIZON     = 1
MODEL_TYPES = ["ridge", "elasticnet", "lgbm"]
MODEL_COLORS = {"ridge": "#1f77b4", "elasticnet": "#ff7f0e", "lgbm": "#2ca02c"}
BENCH_COLORS = {"SPY": "#7f7f7f", "XHB": "#d62728", "PAVE": "#9467bd"}

print(f"Output dir: {PROC_DIR}")

## 1  Build / load walk-forward scores and backtest results

In [ ]:
from urbangrowth.modeling.cross_section import run_cross_sectional_model
from urbangrowth.modeling.backtest import (
    construct_portfolio, performance_stats, _portfolio_stats,
    benchmark_comparison, factor_exposures, sector_decomposition,
)
from urbangrowth.modeling._data import load_monthly_returns, build_forward_returns

raw_returns = load_monthly_returns()
fwd_returns = build_forward_returns(raw_returns, [HORIZON])

scores_all   = {}
portfolio_all = {}

for mt in MODEL_TYPES:
    sp = PROC_DIR / f"model_scores_{mt}_h{HORIZON}.parquet"
    if sp.exists():
        scores_all[mt] = pd.read_parquet(sp)
        print(f"[{mt}] Loaded scores: {len(scores_all[mt]):,} rows")
    else:
        print(f"[{mt}] Computing walk-forward scores...")
        scores_all[mt] = run_cross_sectional_model(model_type=mt, horizon=HORIZON)
        print(f"[{mt}] Done: {len(scores_all[mt]):,} rows")

    if scores_all[mt].empty:
        print(f"[{mt}] WARNING: no scores — skipping")
        continue

    bp = PROC_DIR / f"backtest_{mt}_h{HORIZON}.parquet"
    if bp.exists():
        portfolio_all[mt] = pd.read_parquet(bp)
        print(f"[{mt}] Loaded portfolio: {len(portfolio_all[mt])} months")
    else:
        portfolio_all[mt] = construct_portfolio(scores_all[mt], fwd_returns, horizon=HORIZON, tc_bps=10.0)
        portfolio_all[mt].to_parquet(bp, index=False)
        print(f"[{mt}] Constructed portfolio: {len(portfolio_all[mt])} months")

## 2  Performance statistics

In [ ]:
stat_rows = []
for mt, port in portfolio_all.items():
    s = _portfolio_stats(port)
    s["model"] = mt
    stat_rows.append(s)

stats_df = pd.DataFrame(stat_rows).set_index("model")
fmt = {
    "cagr":         "{:.1%}",
    "ann_vol":      "{:.1%}",
    "sharpe":       "{:.2f}",
    "max_drawdown": "{:.1%}",
    "hit_rate":     "{:.1%}",
    "avg_turnover": "{:.1%}",
    "avg_tc_bps":   "{:.1f}",
    "n_months":     "{:.0f}",
}

display_df = stats_df.copy()
for col, f in fmt.items():
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f.format(x) if pd.notna(x) else "—")
display(display_df)

## 3  Equity curve

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

for mt, port in portfolio_all.items():
    port = port.sort_values("period")
    cum  = (1 + port.set_index("period")["ls_ret"]).cumprod()
    ax.plot(cum.index, cum.values, label=f"{mt.upper()} (net TC)",
            color=MODEL_COLORS[mt], linewidth=2)

# Benchmarks
if not raw_returns.empty:
    bench = (
        raw_returns[raw_returns["symbol"].isin(("SPY", "XHB", "PAVE"))]
        .pivot(index="date", columns="symbol", values="monthly_ret")
        .sort_index()
    )
    if portfolio_all:
        earliest_port = min(p["period"].min() for p in portfolio_all.values())
        bench = bench[bench.index >= earliest_port]
    for sym in ("SPY", "XHB", "PAVE"):
        if sym not in bench.columns:
            continue
        cum_b = (1 + bench[sym].dropna()).cumprod()
        ax.plot(cum_b.index, cum_b.values, label=sym,
                color=BENCH_COLORS[sym], linewidth=1.2, linestyle="--", alpha=0.8)

ax.axhline(1.0, color="black", linewidth=0.8, linestyle=":")
ax.set_title("Cumulative Return — L/S Model vs Benchmarks", fontsize=13)
ax.set_ylabel("Growth of $1")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4  Drawdown analysis

In [ ]:
fig, axes = plt.subplots(len(portfolio_all), 1,
                         figsize=(13, 3 * max(len(portfolio_all), 1)),
                         sharex=True)
if len(portfolio_all) == 1:
    axes = [axes]

for ax, (mt, port) in zip(axes, portfolio_all.items()):
    port = port.sort_values("period")
    ls   = port.set_index("period")["ls_ret"]
    cum  = (1 + ls).cumprod()
    dd   = (cum / cum.cummax() - 1) * 100
    ax.fill_between(dd.index, dd.values, 0, color=MODEL_COLORS[mt], alpha=0.5)
    ax.plot(dd.index, dd.values, color=MODEL_COLORS[mt], linewidth=1)
    ax.set_ylabel("Drawdown (%)")
    ax.set_title(f"{mt.upper()} — Underwater Chart")
    ax.grid(alpha=0.3)
    max_dd = dd.min()
    ax.annotate(f"MaxDD {max_dd:.1f}%", xy=(0.02, 0.1), xycoords="axes fraction",
                fontsize=9, color="darkred")

plt.tight_layout()
plt.show()

## 5  SHAP feature importance (LightGBM)

Re-fit a single LightGBM on all in-sample data for interpretability.  
This is **not** the walk-forward model — it shows globally which features
the model found predictive.

In [ ]:
try:
    import lightgbm as lgb
    import shap
    from urbangrowth.modeling.cross_section import build_model_panel
    from urbangrowth.modeling._data import (
        build_momentum_signal, build_volume_signal, list_signal_sources, load_signal_panel
    )
    from sklearn.impute import SimpleImputer

    sources     = list_signal_sources()
    all_signals = pd.concat([load_signal_panel(s) for s in sources], ignore_index=True) if sources else pd.DataFrame()
    momentum    = build_momentum_signal(raw_returns)
    volume      = build_volume_signal()

    if not all_signals.empty:
        panel = build_model_panel(all_signals, momentum, volume)
        id_cols = ["period", "symbol"]
        feature_cols = [c for c in panel.columns if c not in id_cols]

        fwd_h = fwd_returns[fwd_returns["horizon"] == HORIZON][["symbol", "date", "fwd_return"]].rename(columns={"date": "period"})
        data  = panel.merge(fwd_h, on=["symbol", "period"], how="inner")

        X = data[feature_cols].values.astype(float)
        y = data["fwd_return"].values.astype(float)
        valid = np.isfinite(y)
        X, y  = X[valid], y[valid]

        imp = SimpleImputer(strategy="constant", fill_value=0.0)
        X   = imp.fit_transform(X)

        final_lgbm = lgb.LGBMRegressor(
            n_estimators=200, num_leaves=15, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1,
        )
        final_lgbm.fit(X, y)

        explainer = shap.TreeExplainer(final_lgbm)
        # Use a random 2000-row sample to keep SHAP tractable
        rng = np.random.default_rng(42)
        idx = rng.choice(len(X), size=min(2000, len(X)), replace=False)
        sv  = explainer.shap_values(X[idx])

        mean_abs_shap = np.abs(sv).mean(axis=0)
        importance_df = (
            pd.Series(mean_abs_shap, index=feature_cols)
            .sort_values(ascending=False)
            .head(20)
        )

        fig, ax = plt.subplots(figsize=(10, 6))
        importance_df[::-1].plot.barh(ax=ax, color="steelblue")
        ax.set_title("Top-20 Features — Mean |SHAP| (LightGBM, full-sample fit)", fontsize=12)
        ax.set_xlabel("Mean |SHAP value|")
        ax.grid(alpha=0.3, axis="x")
        plt.tight_layout()
        plt.show()

        print("\nTop 10 features by mean |SHAP|:")
        print(importance_df.head(10).to_string())
    else:
        print("No signal data available — run 'ug signals build' first")

except ImportError as e:
    print(f"LightGBM or SHAP not installed: {e}")
    print("Install with: pip install lightgbm shap")

## 6  Out-of-sample performance 2024–2025

In [ ]:
oos_start = pd.Timestamp("2024-01-01")
oos_rows  = []

for mt, port in portfolio_all.items():
    oos = port[port["period"] >= oos_start]
    if len(oos) < 3:
        print(f"[{mt}] Insufficient OOS data (n={len(oos)})")
        continue
    s = performance_stats(oos.set_index("period")["ls_ret"])
    s["model"] = mt
    s["period"] = f"{oos['period'].min().strftime('%Y-%m')} – {oos['period'].max().strftime('%Y-%m')}"
    oos_rows.append(s)

if oos_rows:
    oos_df = pd.DataFrame(oos_rows).set_index("model")
    fmt_oos = {"cagr": "{:.1%}", "ann_vol": "{:.1%}", "sharpe": "{:.2f}",
               "max_drawdown": "{:.1%}", "hit_rate": "{:.1%}", "n_months": "{:.0f}"}
    ddf = oos_df.copy()
    for col, f in fmt_oos.items():
        if col in ddf.columns:
            ddf[col] = ddf[col].apply(lambda x: f.format(x) if pd.notna(x) else "—")
    print(f"Out-of-sample (2024–2025):")
    display(ddf)
else:
    print("No OOS data yet — backtest horizon ends before 2024")

# OOS equity curve
if oos_rows:
    fig, ax = plt.subplots(figsize=(11, 4))
    for mt, port in portfolio_all.items():
        oos = port[port["period"] >= oos_start].sort_values("period")
        if len(oos) < 3:
            continue
        cum = (1 + oos.set_index("period")["ls_ret"]).cumprod()
        ax.plot(cum.index, cum.values, label=mt.upper(), color=MODEL_COLORS[mt], linewidth=2)
    ax.axhline(1.0, color="black", linewidth=0.8, linestyle=":")
    ax.set_title("Out-of-Sample Equity Curve (2024–2025)", fontsize=12)
    ax.set_ylabel("Growth of $1")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7  Sector decomposition

In [ ]:
best_model = max(portfolio_all, key=lambda mt: portfolio_all[mt].set_index("period")["ls_ret"].mean()) if portfolio_all else None

if best_model and not scores_all[best_model].empty:
    sec_df = sector_decomposition(scores_all[best_model], fwd_returns, horizon=HORIZON)
    if not sec_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        for ax, side in zip(axes, ["long", "short"]):
            sub = sec_df[sec_df["side"] == side].sort_values("avg_contribution", ascending=True)
            colors = ["#2ca02c" if v > 0 else "#d62728" for v in sub["avg_contribution"]]
            ax.barh(sub["sector"], sub["avg_contribution"] * 100, color=colors)
            ax.axvline(0, color="black", linewidth=0.8)
            ax.set_title(f"{side.capitalize()} Leg — Avg Monthly Contribution (bps)")
            ax.set_xlabel("Contribution (% × 100)")
            ax.grid(alpha=0.3, axis="x")
        plt.suptitle(f"Sector Attribution — {best_model.upper()} Model", fontsize=13)
        plt.tight_layout()
        plt.show()
        display(sec_df)
else:
    print("No sector decomposition available")

## 8  Factor exposure (Fama-French 3 + Momentum)

Regresses L/S returns on Mkt-RF, SMB, HML, Mom.  
Requires internet access + `pandas_datareader`.

In [ ]:
for mt, port in portfolio_all.items():
    bp = PROC_DIR / f"backtest_{mt}_h{HORIZON}_factors.parquet"
    if bp.exists():
        ff_df = pd.read_parquet(bp)
    else:
        ls = port.set_index("period")["ls_ret"]
        ff_df = factor_exposures(ls)
        if ff_df is not None:
            ff_df.to_parquet(bp, index=False)

    if ff_df is None or ff_df.empty:
        print(f"[{mt}] Factor data unavailable")
        continue

    print(f"\n{'='*50}")
    print(f"  {mt.upper()} — Factor Regression (HAC s.e., 3 Newey-West lags)")
    print(f"{'='*50}")
    dsp = ff_df.copy()
    dsp["significant"] = dsp["pvalue"] < 0.05
    dsp["beta"]   = dsp["beta"].map("{:.4f}".format)
    dsp["tstat"]  = dsp["tstat"].map("{:.2f}".format)
    dsp["pvalue"] = dsp["pvalue"].map("{:.3f}".format)
    display(dsp)

# Bar chart: factor betas
if portfolio_all:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = None
    for mt in portfolio_all:
        bp = PROC_DIR / f"backtest_{mt}_h{HORIZON}_factors.parquet"
        if not bp.exists():
            continue
        ff = pd.read_parquet(bp).set_index("factor")
        factors = [f for f in ["Mkt-RF", "SMB", "HML", "Mom"] if f in ff.index]
        if not factors:
            continue
        betas = ff.loc[factors, "beta"]
        if x is None:
            x = np.arange(len(factors))
        ax.bar(x + list(portfolio_all.keys()).index(mt) * 0.25,
               betas.values, width=0.22, label=mt.upper(), color=MODEL_COLORS[mt], alpha=0.8)
    if x is not None:
        ax.set_xticks(x + 0.25)
        ax.set_xticklabels(factors)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title("Factor Loadings — All Models", fontsize=12)
        ax.set_ylabel("Beta")
        ax.legend()
        ax.grid(alpha=0.3, axis="y")
        plt.tight_layout()
        plt.show()

## 9  Benchmark comparison table

In [ ]:
if portfolio_all:
    best_port = portfolio_all[max(portfolio_all, key=lambda m: portfolio_all[m].set_index("period")["ls_ret"].mean())]
    bench_df  = benchmark_comparison(best_port, raw_returns)
    fmts = {"cagr": "{:.1%}", "ann_vol": "{:.1%}", "sharpe": "{:.2f}",
            "max_drawdown": "{:.1%}", "hit_rate": "{:.1%}", "n_months": "{:.0f}"}
    ddf = bench_df.copy()
    for col, f in fmts.items():
        if col in ddf.columns:
            ddf[col] = ddf[col].apply(lambda x: f.format(x) if pd.notna(x) else "—")
    display(ddf.set_index("entity"))

## 10  Honest assessment

### What works
- Ridge and ElasticNet provide interpretable linear cross-sections; L/S alpha can be tested
  against FF3+MOM factors to see if it's genuine or latent factor loading.
- LightGBM + SHAP surfaces non-linear signal interactions (permit surge × FERC queue × city CAI).
- Walk-forward design is genuinely out-of-sample: no future data leaks into any prediction.

### Sample-size limits (primary caveat)
- **Universe: ~50 stocks.** Each quintile ≈ 10 names. Standard errors on Sharpe and alpha are
  large — a t-stat of 2.0 on ~80 months is approximately `2.0 / sqrt(80) ≈ 0.22` standard
  deviations wide as a confidence interval on the underlying Sharpe.  Treat point estimates
  as directional indicators, not precise parameters.
- A minimum of 5–7 years of data is typical for meaningful factor inference; signals built
  post-2015 leave limited hold-out history.

### Signal construction caveats
- **National signals (permits, FRED) are uniform per sector** — they're multiplied by sector
  sensitivity weights to create cross-sectional variance, but the dispersion is synthetic.
  Geographically concentrated names (e.g., AZ-heavy homebuilders) benefit more from
  state-level signals; national aggregates are a weaker version for diversified names.
- **Recipient-to-ticker mapping is incomplete.** USASpending subsidiaries and JVs are missed
  unless listed explicitly in `TICKER_RECIPIENT_MAP`. False-negatives suppress signal for
  diversified contractors (e.g., large civil companies with multiple operating subsidiaries).
- **FERC queue structural breaks.** ISOs periodically revise queue methodology (PJM 2022
  queue reform, MISO cluster studies). These create level-shifts that look like signal but
  are reporting artifacts. Queue vintage dates should be checked before extending history.

### Endogeneity and look-ahead
- **FRED construction spending** (TLNRESCONS, PRRESCONS) is revised and partially
  forward-looking in how markets price it — strong series IC may partly reflect that
  institutional investors use the same macro data.  The incremental value over a
  market-implicit estimate is unknown without access to real-time vintage data.
- All signals apply a **1-month implementation lag** to avoid month-end overlap with
  reported returns, but intra-month news effects are not controlled.

### Model fit
- With O(50) stocks and O(100+) features, **regularisation is load-bearing**:
  Ridge α and ElasticNet (α, l1_ratio) are set at fixed defaults and should be
  cross-validated (e.g., expanding purged-CV) before drawing conclusions.
- LightGBM `num_leaves=15` limits depth; trees are shallow by design to avoid
  overfitting thin cross-sections.  SHAP importances from the full-data fit are
  illustrative only — walk-forward feature importance would differ month-by-month.